# Detecção de fraudes em transações de cartão

Este notebook foi feito como projeto da trilha de Python da DIO.

O objetivo é treinar alguns modelos para tentar identificar transações fraudulentas em uma base muito desbalanceada. Como existem pouquíssimas fraudes, a acurácia sozinha não ajuda muito. Por isso eu vou prestar mais atenção em **recall, precisão e F1 da classe de fraude**.

## 1. Bibliotecas

Usei pandas e numpy para trabalhar com os dados, scikit-learn para preparar e avaliar os modelos, XGBoost como terceiro modelo e SHAP para entender melhor as previsões.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    classification_report, confusion_matrix,
    precision_recall_curve, precision_recall_fscore_support,
    roc_auc_score, average_precision_score,
    RocCurveDisplay, PrecisionRecallDisplay
)

from xgboost import XGBClassifier
import shap

pd.set_option("display.max_columns", 40)
RANDOM_STATE = 42

## 2. Carregando a base

A base original é a conhecida **Credit Card Fraud Detection**, com transações reais anonimizadas. O CSV não fica salvo no repositório. Ele é carregado por link quando o notebook roda.

In [ ]:
DATA_URL = "https://storage.googleapis.com/download.tensorflow.org/data/creditcard.csv"

df = pd.read_csv(DATA_URL)
print("Formato da base:", df.shape)
df.head()

## 3. Conferindo o desbalanceamento

Primeiro eu quis ver quantas transações são normais e quantas são fraude. Coloquei o gráfico em escala log porque, sem isso, a barra das fraudes quase não aparece.

In [ ]:
print("Valores ausentes:", int(df.isna().sum().sum()))
print("\nQuantidade por classe:")
print(df["Class"].value_counts())
print("\nPercentual por classe:")
print((df["Class"].value_counts(normalize=True) * 100).round(4))

contagem = df["Class"].value_counts().sort_index()
plt.figure(figsize=(6, 4))
plt.bar(["Normal", "Fraude"], contagem.values)
plt.yscale("log")
plt.title("Distribuição das classes (escala log)")
plt.ylabel("Quantidade de transações")
plt.show()

## 4. Preparação dos dados

Criei duas variáveis simples:

- **LogAmount**: log do valor da compra, para diminuir o efeito de valores muito altos;
- **Hour**: hora aproximada da transação a partir da coluna Time.

Depois separei treino, validação e teste usando `stratify`, para manter a proporção de fraude em cada parte. A validação ficou separada porque usei ela para escolher o limiar sem olhar o resultado final do teste.

In [ ]:
dados = df.copy()
dados["LogAmount"] = np.log1p(dados["Amount"])
dados["Hour"] = (dados["Time"] // 3600) % 24

X = dados.drop(columns="Class")
y = dados["Class"]

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, stratify=y, random_state=RANDOM_STATE
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, stratify=y_temp, random_state=RANDOM_STATE
)

colunas_escala = ["Time", "Amount", "LogAmount", "Hour"]
scaler = StandardScaler()

X_train = X_train.copy()
X_val = X_val.copy()
X_test = X_test.copy()

X_train[colunas_escala] = scaler.fit_transform(X_train[colunas_escala])
X_val[colunas_escala] = scaler.transform(X_val[colunas_escala])
X_test[colunas_escala] = scaler.transform(X_test[colunas_escala])

print("Treino:", X_train.shape, "| fraudes:", int(y_train.sum()))
print("Validação:", X_val.shape, "| fraudes:", int(y_val.sum()))
print("Teste:", X_test.shape, "| fraudes:", int(y_test.sum()))

## 5. Treinando os modelos

Comparei três modelos:

1. Regressão Logística;
2. Random Forest;
3. XGBoost.

Nos três eu tratei o desbalanceamento com peso maior para a classe de fraude. No XGBoost calculei esse peso pela quantidade de exemplos normais dividida pela quantidade de fraudes.

In [ ]:
negativos, positivos = np.bincount(y_train)
peso_xgb = negativos / positivos

modelos = {
    "Regressão Logística": LogisticRegression(
        class_weight="balanced",
        max_iter=1000,
        random_state=RANDOM_STATE
    ),
    "Random Forest": RandomForestClassifier(
        n_estimators=120,
        max_depth=12,
        min_samples_leaf=2,
        class_weight="balanced_subsample",
        n_jobs=-1,
        random_state=RANDOM_STATE
    ),
    "XGBoost": XGBClassifier(
        n_estimators=220,
        max_depth=4,
        learning_rate=0.06,
        subsample=0.85,
        colsample_bytree=0.85,
        scale_pos_weight=peso_xgb,
        eval_metric="logloss",
        n_jobs=2,
        random_state=RANDOM_STATE
    )
}

for nome, modelo in modelos.items():
    print("Treinando:", nome)
    modelo.fit(X_train, y_train)

print("Modelos treinados.")

## 6. Ajustando o limiar de decisão

O padrão costuma ser marcar fraude quando a probabilidade passa de 0,50. Eu preferi testar outros limiares na base de validação.

A regra que usei foi simples: tentar manter pelo menos **85% de recall** na validação e, entre esses pontos, pegar o que tiver melhor precisão. Se nenhum ponto atingir isso, o código usa o maior F1.

In [ ]:
def escolher_limiar(y_real, probabilidades, recall_minimo=0.85):
    precisao, recall, limiares = precision_recall_curve(y_real, probabilidades)

    precisao = precisao[:-1]
    recall = recall[:-1]

    candidatos = np.where(recall >= recall_minimo)[0]

    if len(candidatos) > 0:
        indice = candidatos[np.argmax(precisao[candidatos])]
    else:
        f1 = 2 * (precisao * recall) / (precisao + recall + 1e-9)
        indice = int(np.nanargmax(f1))

    return float(limiares[indice])


def avaliar_modelo(nome, modelo, X_base, y_base, limiar):
    proba = modelo.predict_proba(X_base)[:, 1]
    pred = (proba >= limiar).astype(int)

    precisao, recall, f1, _ = precision_recall_fscore_support(
        y_base, pred, average="binary", zero_division=0
    )

    return {
        "Modelo": nome,
        "Limiar": limiar,
        "Precisão fraude": precisao,
        "Recall fraude": recall,
        "F1 fraude": f1,
        "ROC-AUC": roc_auc_score(y_base, proba),
        "PR-AUC": average_precision_score(y_base, proba)
    }


limiares = {}
resultados = []

for nome, modelo in modelos.items():
    proba_val = modelo.predict_proba(X_val)[:, 1]
    limiar = escolher_limiar(y_val, proba_val, recall_minimo=0.85)
    limiares[nome] = limiar
    resultados.append(
        avaliar_modelo(nome, modelo, X_test, y_test, limiar)
    )

comparacao = pd.DataFrame(resultados)
comparacao.round(4)

## 7. Curvas ROC e Precisão x Recall

A curva ROC ajuda na comparação geral, mas em uma base tão desbalanceada eu acho a curva de precisão e recall mais fácil de interpretar para o objetivo do projeto.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
for nome, modelo in modelos.items():
    proba = modelo.predict_proba(X_test)[:, 1]
    RocCurveDisplay.from_predictions(y_test, proba, name=nome, ax=ax)

ax.set_title("Curva ROC")
plt.show()

fig, ax = plt.subplots(figsize=(7, 5))
for nome, modelo in modelos.items():
    proba = modelo.predict_proba(X_test)[:, 1]
    PrecisionRecallDisplay.from_predictions(y_test, proba, name=nome, ax=ax)

ax.set_title("Curva Precisão x Recall")
plt.show()

## 8. Relatório do modelo escolhido

Para a análise final, peguei o modelo com melhor F1 entre os que conseguiram pelo menos 80% de recall no teste. Se nenhum chegar nisso, fica o maior F1.

Não estou usando acurácia para escolher o modelo, porque nesse problema ela pode ficar perto de 100% mesmo quando o modelo quase não encontra fraude.

In [ ]:
validos = comparacao[comparacao["Recall fraude"] >= 0.80]

if len(validos) == 0:
    melhor_nome = comparacao.sort_values("F1 fraude", ascending=False).iloc[0]["Modelo"]
else:
    melhor_nome = validos.sort_values("F1 fraude", ascending=False).iloc[0]["Modelo"]

melhor_modelo = modelos[melhor_nome]
melhor_limiar = limiares[melhor_nome]

proba_melhor = melhor_modelo.predict_proba(X_test)[:, 1]
pred_melhor = (proba_melhor >= melhor_limiar).astype(int)

print("Modelo usado na análise final:", melhor_nome)
print("Limiar escolhido:", round(melhor_limiar, 4))
print("\nRelatório de classificação:\n")
print(classification_report(
    y_test,
    pred_melhor,
    target_names=["normal", "fraude"],
    digits=4
))

print("Matriz de confusão:")
print(confusion_matrix(y_test, pred_melhor))

## 9. Importância das variáveis

Para ter uma visão mais simples, comecei olhando a importância das variáveis no Random Forest.

In [ ]:
rf = modelos["Random Forest"]

importancia_rf = pd.Series(
    rf.feature_importances_,
    index=X_train.columns
).sort_values(ascending=False).head(12)

plt.figure(figsize=(8, 5))
importancia_rf.sort_values().plot(kind="barh")
plt.title("Random Forest - variáveis mais importantes")
plt.xlabel("Importância")
plt.show()

importancia_rf.to_frame("importancia")

## 10. Explicando o XGBoost com SHAP

A importância mostra quais variáveis aparecem mais no modelo, mas o SHAP ajuda a ver **como** cada variável empurrou uma previsão para fraude ou para normal.

Usei uma amostra pequena do teste para o gráfico não ficar pesado demais.

In [ ]:
modelo_xgb = modelos["XGBoost"]

amostra_shap = X_test.sample(
    n=min(300, len(X_test)),
    random_state=RANDOM_STATE
)

explainer = shap.TreeExplainer(modelo_xgb)
valores_shap = explainer(amostra_shap)

shap.plots.beeswarm(
    valores_shap,
    max_display=12,
    show=False
)
plt.title("SHAP - impacto das variáveis no XGBoost")
plt.show()

print("Exemplo de explicação para uma transação da amostra:")
shap.plots.waterfall(
    valores_shap[0],
    max_display=12,
    show=False
)
plt.show()

## 11. Resumo

A parte que mais mudou a minha forma de olhar o problema foi perceber que **acurácia alta não significa um bom detector de fraude**.

O que eu priorizei foi:

- manter um recall alto para deixar menos fraudes passarem;
- observar a precisão para não transformar quase tudo em alerta;
- comparar o F1 para ter uma noção do equilíbrio entre os dois;
- ajustar o limiar em vez de aceitar 0,50 automaticamente;
- usar SHAP para tentar entender as decisões do modelo.

Como próximos testes eu faria undersampling e oversampling, além de testar mais combinações de hiperparâmetros.

In [ ]:
resumo_final = comparacao[
    ["Modelo", "Limiar", "Precisão fraude", "Recall fraude", "F1 fraude"]
].copy()

resumo_final = resumo_final.round(4)

print(resumo_final.to_string(index=False))
print("\nModelo escolhido para a análise final:", melhor_nome)
print("Acurácia não foi usada como métrica principal porque a classe de fraude é muito rara.")